# Panduan Colab — hasil tersimpan dan bukti BAB IV

Jalankan **V01 → V02 → V03 → … → V17**. V18 dan V19 adalah lampiran/penutup.
Notebook ini membaca hasil dan gambar yang sudah ada pada commit sumber terkunci. **Tidak melakukan scraping, labeling, training atau mengeksekusi source cell notebook penelitian.** CPU cukup; tidak perlu token API atau mount Drive.

Output sumber lama akan diberi label **ARSIP**. Menampilkan output arsip tidak sama dengan menjalankan ulang penelitian. Status akademik PROVISIONAL; gap provenance AI dan empat isu metodologis tetap terbuka.

Nomor V adalah langkah notebook pendamping. Nomor cell asli ada pada panduan Markdown dan CSV pendamping; nomor itu menghitung semua cell, termasuk Markdown, dari atas mulai 1. Angka `In[...]` tidak dipakai sebagai identitas cell.

## V01 — Ambil salinan sumber GitHub
Salinan dibuat di penyimpanan runtime. Jika folder sudah ada, notebook memeriksa commitnya dan tidak mereset isinya.

In [ ]:
# V01 — SETUP SUMBER TERKUNCI
from pathlib import Path
import os, subprocess, sys, json, csv, hashlib, base64, re
SOURCE_COMMIT = "d344822ab6d6aaca76b3c658c101c54e4b25bac1"
ROOT = Path(os.environ.get("SENTIMEN_REPO_ROOT", "/content/penelitian_sentimen_sumber")).resolve()
REPO_URL = "https://github.com/YusakSR/Penelitian-Sentimen-Bencana-Nasional.git"
if not ROOT.exists():
    subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "-c", "core.autocrlf=false", "checkout", "--detach", SOURCE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_COMMIT:
    raise RuntimeError("Folder sudah ada pada commit lain. Gunakan folder runtime baru; sumber tidak di-reset otomatis.")
print("Sumber:", SOURCE_COMMIT)
print("Folder:", ROOT)
print("Mode: membaca hasil tersimpan. Scraping/training: tidak dijalankan.")

## V02 — Periksa integritas dan siapkan pembaca
Pemeriksa memvalidasi bukti dari commit sumber. Hasilnya dapat menyebut gap provenance AI; itu tetap dicatat, tidak diisi otomatis.

In [ ]:
# V02 — INTEGRITAS DAN PEMBACA
import pandas as pd
from IPython.display import display, Markdown, Image
result = subprocess.run([sys.executable, "-B", str(ROOT / "07_Documentation/scripts/verify_cleanup_lineage.py")], check=True, capture_output=True, text=True)
integrity = json.loads(result.stdout)
print("Integritas:", integrity["status"])
print("Observasi terlacak:", integrity["observations_traced_to_raw"])
print("Notebook penelitian/backup identik:", integrity["research_and_backup_notebooks_byte_identical"])
print("Gap:", integrity["provenance_gap"])
def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8-sig"))
def table(relative):
    return pd.read_csv(ROOT / relative, encoding="utf-8-sig", dtype=str, keep_default_na=False)
def archived_text(notebook, number):
    cell = load_json(notebook)["cells"][number - 1]
    heading = next((line for line in "".join(cell["source"]).splitlines() if line.strip()), "")
    print("ARSIP —", notebook, "— cell asli", number, "— ID", cell.get("id", ""))
    print(heading)
    for output in cell.get("outputs", []):
        if output.get("output_type") == "stream":
            text = "".join(output.get("text", []))
            print(re.sub(r"\x1b\[[0-9;]*m", "", text))
        elif output.get("output_type") == "error":
            print("ERROR ARSIP:", output.get("ename"), output.get("evalue"))
    print("Teks di atas adalah keluaran tersimpan, bukan eksekusi ulang cell sumber.")
figures = table("06_Result/outputs/integrated_report/final_main_figure_order_10.csv")
def show_main(number):
    row = figures.iloc[number - 1]
    data = (ROOT / row["original_path"]).read_bytes()
    if hashlib.sha256(data).hexdigest() != row["sha256"]:
        raise ValueError("SHA gambar berbeda: " + row["original_path"])
    display(Markdown("### " + row["figure_number"] + " — " + row["caption"]))
    display(Image(data=data))
    print("PNG asli:", row["original_path"])
    print("Catatan:", row["interpretation_note"])

## V03 — Fase 01: hasil pengumpulan dan screening
Rujukan sumber: full collection cell 41 dan 45.

In [ ]:
# V03 — RAW DAN SCREENING FINAL
raw = load_json("01_Raw Data/outputs/reports/batch7/final_collection/current_manifest.json")
review = load_json("01_Raw Data/outputs/reports/batch7/manual_reconciliation/Batch7_Reconciliation_Summary.json")
display(pd.DataFrame([{"SAVED task":raw["saved_tasks"], "Raw record":raw["raw_total"], "Kemunculan tweet asli":raw["tweet_rows"], "Mock":raw["mock_rows"], "ID unik":raw["unique_ids"], "INCLUDE":review["include"], "EXCLUDE":review["exclude"]}]))
archived_text("01_Raw Data/full_scraping_raw_data.ipynb", 41)
archived_text("01_Raw Data/full_scraping_raw_data.ipynb", 45)

## V04 — Fase 02: cleaning dan seleksi kandidat
Rujukan sumber: cell 3, 4, 7, 10 dan 11. Tabel contoh adalah kutipan teks tersimpan; tidak menormalisasi ulang teks.

In [ ]:
# V04 — CLEANING FINAL
clean = table("02_Cleaning/outputs/reconciliation/part8_content_reconciled.csv")
print("Master cleaning:", len(clean), "ID unik:", clean["id"].nunique())
display(clean["model_selection_status"].value_counts().rename_axis("status").reset_index(name="jumlah"))
print("Contoh lima record dari artefak final:")
display(clean[["id", "text_original", "text_semantic_verified", "text_linguistic"]].head(5))
archived_text("02_Cleaning/02_text_cleaning.ipynb", 10)
archived_text("02_Cleaning/02_text_cleaning.ipynb", 11)

## V05 — Fase 03: labeling dan perbedaan versi label
Rujukan sumber: labeling cell 14–15; modeling cell 6. Label fase 03 tidak dicampur dengan label akhir fase 04.

In [ ]:
# V05 — LABEL FINAL DAN PROVENANCE
labeling = table("03_Labeling/outputs/final_labeling/modeling_labeled_348.csv")
modeling = table("04_Modeling/outputs/leakage_grouping/modeling_labeled_348_grouped.csv")
dist = pd.concat([labeling["sentiment_label"].value_counts().rename("fase03"), modeling["sentiment_label"].value_counts().rename("fase04")], axis=1).reindex(["NETRAL","POSITIF","NEGATIF"])
display(dist)
display(table("04_Modeling/outputs/label_adjudication/label_adjudication_changelog.csv"))
ai = table("03_Labeling/outputs/hybrid_labeling/ai_predictions_251.csv")
print("Prediksi AI yang benar-benar tersedia:", len(ai), "baris; nama file bukan bukti 251 prediksi lengkap.")
print("Status asal label AI/manual memerlukan bukti penerimaan/review yang lengkap.")
archived_text("03_Labeling/03_sentiment_labeling.ipynb", 15)

## V06 — Fase 04: dataset, pembagian evaluasi dan metrik final
Metrik berasal dari file audit; tidak melatih model. Sumber asli cell 21 menyimpan error rekonsiliasi; gunakan hasil final cell 24/35/37 untuk BAB IV.

In [ ]:
# V06 — MODELING FINAL TANPA TRAINING
grouped = table("04_Modeling/outputs/leakage_grouping/modeling_labeled_348_grouped.csv")
folds = table("04_Modeling/outputs/cv_split_final/cv_final_assignments_348.csv")
print("Observasi:", len(grouped), "Leakage groups:", grouped["leakage_group_id"].nunique())
display(folds["validation_fold"].value_counts().sort_index().rename_axis("validation_fold").reset_index(name="jumlah"))
display(table("06_Result/outputs/consolidated_results/consolidated_model_evaluation.csv"))
display(table("06_Result/outputs/consolidated_results/model_error_analysis_by_class.csv"))
print("Evaluasi: 3-fold SGKF OOF; bukan independent holdout test set.")
archived_text("04_Modeling/04_modeling.ipynb", 24)

## V07 — Fase 05: temporal dengan label anotasi terkunci
Sumber cell 2–5. Waktu WIB; tanggal tanpa observasi bukan bukti bahwa diskusi publik tidak ada.

In [ ]:
# V07 — TEMPORAL FINAL
temporal = load_json("05_Temporal/outputs/daily_aggregation/part3_daily_aggregation_summary.json")
display(pd.DataFrame(temporal["coverage_by_case"]))
print("Jumlah tweet:", temporal["total_aggregated_tweets"], "Hari kalender:", temporal["daily_calendar_rows"])
print("Sumber label:", temporal["label_source"], "Zona waktu:", temporal["timezone"])
display(table("05_Temporal/outputs/daily_aggregation/daily_sentiment_counts_20_days.csv"))

## V08 — Gambar 4.1: Perbandingan metrik evaluasi out-of-fold antara baseline TF-IDF + Logistic Regression dan LSTM.
Sumber: 04_Modeling, cell asli 30. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V08 — GAMBAR UTAMA 1
show_main(1)

## V09 — Gambar 4.2: Matriks kebingungan prediksi out-of-fold model LSTM pada tiga kelas sentimen.
Sumber: 04_Modeling, cell asli 29. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V09 — GAMBAR UTAMA 2
show_main(2)

## V10 — Gambar 4.3: Perbandingan F1-score per kelas sentimen antara baseline dan LSTM berdasarkan prediksi OOF.
Sumber: 04_Modeling, cell asli 31. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V10 — GAMBAR UTAMA 3
show_main(3)

## V11 — Gambar 4.4: Jumlah tweet menurut kelas sentimen per hari untuk kasus Lombok 2018 (WIB).
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V11 — GAMBAR UTAMA 4
show_main(4)

## V12 — Gambar 4.5: Jumlah tweet menurut kelas sentimen per hari untuk kasus Palu 2018 (WIB).
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V12 — GAMBAR UTAMA 5
show_main(5)

## V13 — Gambar 4.6: Jumlah tweet menurut kelas sentimen per hari untuk kasus Sumatera 2025 (WIB).
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V13 — GAMBAR UTAMA 6
show_main(6)

## V14 — Gambar 4.7: Persentase label sentimen harian pada sampel tweet Lombok 2018 beserta ukuran sampel harian.
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V14 — GAMBAR UTAMA 7
show_main(7)

## V15 — Gambar 4.8: Persentase label sentimen harian pada sampel tweet Palu 2018 beserta ukuran sampel harian.
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V15 — GAMBAR UTAMA 8
show_main(8)

## V16 — Gambar 4.9: Persentase label sentimen harian pada sampel tweet Sumatera 2025 beserta ukuran sampel harian.
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V16 — GAMBAR UTAMA 9
show_main(9)

## V17 — Gambar 4.10: Komposisi label sentimen manusia pada sampel tweet Lombok 2018, Palu 2018, dan Sumatera 2025.
Sumber: 05_Temporal, cell asli 4. Unduh PNG asli untuk naskah; screenshot hanya bila diperlukan.

In [ ]:
# V17 — GAMBAR UTAMA 10
show_main(10)

## V18 — Tujuh grafik lampiran (opsional)
Loss/accuracy training bukan performa OOF; hari relatif 0 berarti awal observasi.

In [ ]:
# V18 — GAMBAR LAMPIRAN OPSIONAL
appendix_paths = [
    "04_Modeling/outputs/lstm_visualizations/01_training_loss_per_fold.png",
    "04_Modeling/outputs/lstm_visualizations/02_training_accuracy_per_fold.png",
    "04_Modeling/outputs/lstm_visualizations/03_macro_f1_per_fold.png",
    "04_Modeling/outputs/lstm_visualizations/07_vocabulary_size_per_fold.png",
    "04_Modeling/outputs/lstm_visualizations/08_token_count_distribution.png",
    "04_Modeling/outputs/lstm_visualizations/09_token_count_by_label_boxplot.png",
    "05_Temporal/outputs/timeline_visualizations/07_relative_day_tweet_volume.png",
]
locked = load_json("06_Result/outputs/cross_phase_integrity/part1_cross_phase_integrity_audit.json")["figure_sha256"]
for relative in appendix_paths:
    data = (ROOT / relative).read_bytes()
    if hashlib.sha256(data).hexdigest() != locked[relative]:
        raise ValueError("SHA gambar lampiran berbeda: " + relative)
    print(relative)
    display(Image(data=data))

## V19 — Fase 06–07: ringkasan, isu terbuka dan pemeriksaan penutup
Tabel hasil lebih baik dipindahkan menjadi tabel Word yang dapat diedit. Sumber PDF tetap provisional.

In [ ]:
# V19 — RESULT, DOCUMENTATION DAN PENUTUP
display(table("06_Result/outputs/consolidated_results/dataset_characteristics.csv"))
display(figures[["figure_number","chapter_section","caption","original_path"]])
issues = load_json("06_Result/outputs/integrated_report/open_methodological_issues.json")
display(pd.DataFrame(issues["issues"])[["id","topic","resolved"]])
display(table("07_Documentation/outputs/part2_provenance/lineage_edges.csv"))
final = subprocess.run([sys.executable, "-B", str(ROOT / "07_Documentation/scripts/verify_cleanup_lineage.py")], check=True, capture_output=True, text=True)
print("Integritas setelah penayangan:", json.loads(final.stdout)["status"])
print("Semua hasil yang ditampilkan berasal dari commit sumber", SOURCE_COMMIT)
print("Notebook sumber tidak dijalankan; tidak ada scraping, training, atau perubahan file penelitian.")